# 04 · Build the dataset for the first experiment

Goal: one clean table with exactly the rows, features and target from the README (*Target and features*), split into train / valid / test and saved as Parquet. MLflow will read these files.

| Split | Data | Purpose |
| --- | --- | --- |
| `train` | rides starting 2024-07-01 to 2024-07-24 | fit the model |
| `valid` | rides starting 2024-07-25 to 2024-07-31 | early stopping, choosing hyperparameters |
| `test` | July 2026 | final evaluation, used **once** at the end |

Steps:
1. Read both months **lazily** and apply the row rules while reading
2. **Split first**, before any feature is computed
3. Target and features from the timetable
4. Public holidays and weather forecast
5. Keep only the columns we use, check, save

In [1]:
from datetime import date

import polars as pl

from db_delay_mlops.features.holidays import add_is_holiday
from db_delay_mlops.features.weather import add_weather
from db_delay_mlops.paths import DATA_DIR, STATIONS_FILE, TRAIN_TYPE_GROUPS_FILE

HF_DIR = DATA_DIR / "raw" / "hf" / "monthly_processed_data"
OUT_DIR = DATA_DIR / "processed" / "first_experiment"  # data/ is in .gitignore

MONTHS = ["2024-07", "2026-07"]
TEST_MONTH = "2026-07"
VALID_FROM = date(2024, 7, 25)  # rides starting on or after this day go to `valid`
RAIL_GROUPS = ["long_distance", "regional", "s_bahn"]  # no buses, no unmapped train types

## 1. Read lazily and filter while reading

A month from 2025-11 on has about 15 M rows and 23 columns; WSL has only 7 GB RAM. Therefore:

- `scan_parquet` instead of `read_parquet`: nothing is loaded yet, Polars only builds a plan.
- `select` first: only the columns we need are read from disk.
- `filter` before `collect`: Polars applies the filters while reading, rows of other stations or buses never reach memory.
- One function for one month, then `pl.concat` of both plans and **one** `collect`.

Row rules (README): our 134 rail EVA numbers (146 minus the 12 bus stations), only rail train groups (buses out), planned arrival (not the first stop), not cancelled, stop is in the timetable.

`ride_start` is the planned start of the whole ride, encoded in the `id`: `<ride id>-<YYMMDDHHmm>-<stop number>`. We need it for the split and for `minutes_since_departure`.

In [2]:
stations = pl.read_csv(STATIONS_FILE, schema_overrides={"eva": pl.String})
rail_evas = stations.filter(pl.col("collect"))["eva"]  # 146 EVA numbers minus 12 bus stations = 134
groups = pl.read_csv(TRAIN_TYPE_GROUPS_FILE).select("train_type", train_group="group")

RAW_COLUMNS = [
    "id", "eva", "train_type", "line_number", "final_destination_station", "train_line_station_num",
    "has_plan_record", "arrival_planned_time", "arrival_change_time", "arrival_is_canceled",
]


def scan_month(month: str) -> pl.LazyFrame:
    """Plan for one monthly file: needed columns, row rules, ride start. Nothing is read yet."""
    return (
        pl.scan_parquet(HF_DIR / f"data-{month}.parquet")
        .select(RAW_COLUMNS)
        .filter(
            pl.col("eva").is_in(rail_evas.implode()),  # only our rail stations
            pl.col("arrival_planned_time").is_not_null(),  # not the first stop of a ride
            ~pl.col("arrival_is_canceled"),  # cancelled arrivals are excluded
            pl.col("has_plan_record"),  # only stops that are in the timetable
        )
        .join(groups.lazy(), on="train_type", how="left")
        .filter(pl.col("train_group").is_in(RAIL_GROUPS))  # buses and unmapped types out
        .with_columns(
            source_month=pl.lit(month),  # which file the row comes from
            ride_start=pl.col("id").str.extract(r"-(\d{10})-\d+$").str.to_datetime("%y%m%d%H%M", time_unit="ns"),
        )
    )


arrivals = pl.concat([scan_month(m) for m in MONTHS]).collect()
print(f"{arrivals.height:,} rows, {arrivals.estimated_size('mb'):.0f} MB in memory")
arrivals.group_by("source_month").len().sort("source_month")

2,732,492 rows, 266 MB in memory


source_month,len
str,u32
"""2024-07""",1405544
"""2026-07""",1326948


## 2. Split first

**Why at the beginning:** everything that is *learned from data* (e.g. later the delay rate of a line, category encodings, how missing values are filled) must only see the training data. If we split at the end, it is easy to compute such a value on all data by accident, and the test score is too optimistic. Splitting first makes that mistake impossible.

**Why by time and not at random:** in production the model predicts the future. A random split would put stops of the same day, and even of the same ride, into train and test.

**Why by ride start and not by arrival time:** a ride that starts at 23:30 on 24 July has stops before and after midnight. Splitting by `ride_start` keeps all stops of one ride in the same split.

**Why a validation set:** LightGBM needs data to decide when to stop training (early stopping), and we will compare several settings. If we did that on the test set, the test set would no longer be independent.

In [3]:
arrivals = arrivals.with_columns(
    split=pl.when(pl.col("source_month") == TEST_MONTH)
    .then(pl.lit("test"))
    .when(pl.col("ride_start").dt.date() >= VALID_FROM)
    .then(pl.lit("valid"))
    .otherwise(pl.lit("train"))
)

# Check: no ride may appear in two splits
ride_key = pl.col("id").str.extract(r"^(.*)-\d{10}-\d+$") + pl.col("ride_start").cast(pl.String)
rides_in_several_splits = arrivals.group_by(ride_key.alias("ride")).agg(pl.col("split").n_unique()).filter(pl.col("split") > 1).height
print("rides in more than one split:", rides_in_several_splits)

arrivals.group_by("split").agg(
    rows=pl.len(),
    first_arrival=pl.col("arrival_planned_time").min(),
    last_arrival=pl.col("arrival_planned_time").max(),
).sort("first_arrival")

rides in more than one split: 0


split,rows,first_arrival,last_arrival
str,u32,datetime[ns],datetime[ns]
"""train""",1120128,2024-06-30 23:18:00,2024-07-25 10:01:00
"""valid""",285416,2024-07-25 00:04:00,2024-07-31 23:59:00
"""test""",1326948,2026-06-30 21:35:00,2026-08-01 00:00:00


## 3. Target and features from the timetable

- `y`: arrival delay ≥ 6 minutes, computed from the arrival columns.
- `hour`, `weekday` (1 = Monday … 7 = Sunday), `month`: from the **planned** arrival.
- `minutes_since_departure`: planned arrival minus planned start of the ride. 0.01 % of the rows are slightly negative (mostly −1 minute, rounding); they are set to 0.
- `line`: `line_number` changed its format. In 2024 DB trains only had digits (`S` + `"1"`, `RE` + `"1"`), in 2026 always a prefix (`"S1"`, `"RE1"`). Without cleaning, `"1"` and `"S1"` would be two different lines. Rule: remove spaces; if only digits are left, put the train type in front. Long-distance trains usually have no line (about 97 % empty); LightGBM handles missing values.

In [4]:
line_clean = pl.col("line_number").str.replace_all(" ", "")

arrivals = arrivals.join(stations.select("eva", "stada_name"), on="eva", how="left").with_columns(
    y=((pl.col("arrival_change_time") - pl.col("arrival_planned_time")).dt.total_minutes() >= 6).cast(pl.Int8),
    hour=pl.col("arrival_planned_time").dt.hour(),
    weekday=pl.col("arrival_planned_time").dt.weekday(),  # 1 = Monday, 7 = Sunday (ISO)
    month=pl.col("arrival_planned_time").dt.month(),
    minutes_since_departure=(pl.col("arrival_planned_time") - pl.col("ride_start")).dt.total_minutes().clip(lower_bound=0),
    line=pl.when(line_clean.str.contains(r"^\d+$")).then(pl.col("train_type") + line_clean).otherwise(line_clean),
)

# Same line, same name in both years?
arrivals.filter(pl.col("train_type") == "S").group_by("source_month").agg(
    pl.col("line").value_counts(sort=True).head(5).alias("top_s_bahn_lines")
)

source_month,top_s_bahn_lines
str,list[struct[2]]
"""2024-07""","[{""S1"",115231}, {""S3"",82353}, … {""S7"",45591}]"
"""2026-07""","[{""S1"",102815}, {""S3"",85778}, … {""S7"",53643}]"


## 4. Public holidays and weather forecast

Both functions only need the time column and the station column; federal state and coordinates come from `config/stations.csv`. `add_weather` loads every month that occurs (here 2024-06, 2024-07, 2026-06, 2026-07) and fetches missing months from Open-Meteo once.

These features are computed row by row and learn nothing from the data, so they cannot leak information between the splits.

In [5]:
arrivals = add_is_holiday(arrivals, time_col="arrival_planned_time", station_col="stada_name")
arrivals = add_weather(arrivals, time_col="arrival_planned_time", station_col="stada_name")

print("holidays:", arrivals["is_holiday"].sum(), "| rows without weather:", arrivals["temperature_2m"].null_count())

holidays: 0 | rows without weather: 0


## 5. Keep only the columns we use, check, save

- **Features and target** exactly as in the README.
- **Metadata** (`id`, `eva`, `arrival_planned_time`, `split`) stays for analysis and joins, but is **not** a feature. The training script will select `FEATURES` explicitly.
- Everything that contains the outcome (`arrival_change_time`, the delay in minutes) is **dropped**, so it cannot end up in the model by accident.

In [6]:
CATEGORICAL_FEATURES = ["stada_name", "line", "final_destination_station", "train_group"]
NUMERIC_FEATURES = [
    "train_line_station_num", "minutes_since_departure", "hour", "weekday", "month",
    "temperature_2m", "precipitation", "weather_code",
]
BOOLEAN_FEATURES = ["is_holiday"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES + BOOLEAN_FEATURES
TARGET = "y"
METADATA = ["id", "eva", "arrival_planned_time", "split"]

dataset = arrivals.select(*METADATA, *FEATURES, TARGET)
print(dataset.shape, f"{dataset.estimated_size('mb'):.0f} MB")
dataset.head()

(2732492, 18) 330 MB


id,eva,arrival_planned_time,split,stada_name,line,final_destination_station,train_group,train_line_station_num,minutes_since_departure,hour,weekday,month,temperature_2m,precipitation,weather_code,is_holiday,y
str,str,datetime[ns],str,str,str,str,str,i32,i64,i8,i8,i8,f64,f64,i16,bool,i8
"""-4428241117209577603-240730230…","""08000260""",2024-07-31 00:16:00,"""valid""","""Würzburg Hbf""","""RB53""","""Würzburg Hbf""","""regional""",14,72,0,3,7,19.5,0.0,0,false,0
"""2233068139550272688-2407302319…","""08000253""",2024-07-31 00:18:00,"""valid""","""Mönchengladbach Hbf""","""RB27""","""Mönchengladbach Hbf""","""regional""",12,59,0,3,7,23.2,0.0,0,false,0
"""6339251453367037033-2407302207…","""08000134""",2024-07-31 00:19:00,"""valid""","""Trier Hbf""","""RB81""","""Trier Hbf""","""regional""",33,132,0,3,7,24.8,0.0,0,false,0
"""601208318005647306-2407302020-…","""08000294""",2024-07-30 22:41:00,"""valid""","""Osnabrück Hbf""","""RE60""","""Rheine""","""regional""",19,141,22,2,7,22.6,0.0,2,false,1
"""-8543024795629287088-240730225…","""08003368""",2024-07-31 00:03:00,"""valid""","""Köln Messe/Deutz""","""RB25""","""Köln Hansaring""","""regional""",12,71,0,3,7,26.0,0.0,0,false,1


In [7]:
# Rows, late share and missing values per split
with pl.Config(tbl_cols=-1, float_precision=3):
    display(
        dataset.group_by("split").agg(
            rows=pl.len(),
            late_share=pl.col("y").mean(),
            share_long_distance=(pl.col("train_group") == "long_distance").mean(),
            missing_line=pl.col("line").is_null().mean(),
            missing_weather=pl.col("temperature_2m").is_null().mean(),
        ).sort("split", descending=True)
    )

# Leakage guard: none of these columns may be in the dataset
forbidden = {"arrival_change_time", "departure_change_time", "delay_in_min", "time", "arrival_is_canceled"}
assert not forbidden & set(dataset.columns), forbidden & set(dataset.columns)
# Every row has a target, and buses are gone
assert dataset["y"].null_count() == 0
assert set(dataset["train_group"].unique()) <= set(RAIL_GROUPS)
print("checks passed")

split,rows,late_share,share_long_distance,missing_line,missing_weather
str,u32,f64,f64,f64,f64
"""valid""",285416,0.197,0.122,0.124,0.000
"""train""",1120128,0.209,0.133,0.134,0.000
"""test""",1326948,0.234,0.115,0.112,0.000


checks passed


In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for split in ["train", "valid", "test"]:
    path = OUT_DIR / f"{split}.parquet"
    dataset.filter(pl.col("split") == split).drop("split").write_parquet(path)
    print(f"{path.relative_to(DATA_DIR.parent)}: {path.stat().st_size / 1e6:.1f} MB")

data/processed/first_experiment/train.parquet: 21.1 MB
data/processed/first_experiment/valid.parquet: 5.5 MB
data/processed/first_experiment/test.parquet: 25.8 MB


## Questions

1. Compare the late share of `train` and `test`. What does the difference mean for the test score of a model trained only on July 2024?
2. `month` is 7 in every row of this experiment. What can the model learn from it here? When does it become useful?
3. `S1` exists in Berlin, Munich, Hamburg and other cities, but these are different lines. What could go wrong, and which other feature helps the model to tell them apart?
4. Why do we keep `id` and `arrival_planned_time` in the files, if they are not features?